# LAB 1 – Bài 2: Titanic

**Sinh viên:** Lê Nguyễn Anh Dự – **MSSV:** 6451071011

Chạy các ô lần lượt. Mỗi ô có output in thông tin sinh viên.


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display
STUDENT_INFO="Thông tin sinh viên: Lê Nguyễn Anh Dự - 6451071011"
sns.set_theme(style="whitegrid")
!pip -q install gdown
import gdown
FILE_ID = "1TCnJqaowkv0caJTJrmjoqb0AB2bMvgxP"
DATA_PATH = Path("titanic.csv")
gdown.download(id=FILE_ID, output=str(DATA_PATH), quiet=False)
if not DATA_PATH.exists(): raise FileNotFoundError("Could not download titanic.csv; check Google Drive sharing permissions.")
df = pd.read_csv(DATA_PATH)
print(f"Downloaded and loaded data from Google Drive | {STUDENT_INFO}")


## Câu 1. Nhập dữ liệu và thống kê thuộc tính


In [ ]:
print(f"Kích thước: {df.shape} | {STUDENT_INFO}")
print(f"Thông tin cột | {STUDENT_INFO}"); df.info()
print(f"Thống kê mô tả | {STUDENT_INFO}"); display(df.describe(include="all").T)


## Câu 2. Top 5 thuộc tính nhiều giá trị rỗng


In [ ]:
missing=pd.DataFrame({"Số giá trị rỗng":df.isna().sum(),"Tỷ lệ rỗng (%)":(df.isna().mean()*100).round(2)}).sort_values("Số giá trị rỗng",ascending=False).head(5)
print(f"Top 5 giá trị rỗng | {STUDENT_INFO}"); display(missing)


## Câu 3. Tuổi, giới tính và sống sót


In [ ]:
fig,ax=plt.subplots(figsize=(9,5)); sns.boxplot(data=df,x="sex",y="age",hue="survived",showfliers=False,ax=ax)
ax.set(title="Tuổi theo giới tính và sống sót",xlabel="Giới tính",ylabel="Tuổi"); ax.legend(title="Sống sót",labels=["Không","Có"]); plt.tight_layout()
print(f"Biểu đồ tuổi × giới tính × sống sót | {STUDENT_INFO}"); plt.show()
sr=df.groupby("sex").survived.mean().mul(100).round(1).rename("Tỷ lệ sống sót (%)")
print(f"Tỷ lệ sống sót theo giới tính | {STUDENT_INFO}"); display(sr.to_frame())
print("Nhận xét: phụ nữ có tỷ lệ sống sót cao hơn nam giới; phân phối tuổi hai nhóm có chồng lấn, nên tuổi không phân tách hoàn toàn người sống sót.")


## Câu 4. Embarked, pclass, sex và sống sót


In [ ]:
q=df.dropna(subset=["embarked"]); s=q.groupby(["sex","pclass","embarked"]).survived.mean().mul(100).round(1).rename("Tỷ lệ sống sót (%)").reset_index()
print(f"Bảng tỷ lệ sống theo giới tính, hạng vé, cảng | {STUDENT_INFO}"); display(s)
fig,axs=plt.subplots(2,3,figsize=(14,8),sharey=True)
for i,sex in enumerate(["female","male"]):
 for j,pc in enumerate([1,2,3]):
  ax=axs[i,j]; z=s[(s.sex==sex)&(s.pclass==pc)]
  sns.barplot(data=z,x="embarked",y="Tỷ lệ sống sót (%)",order=["C","Q","S"],color="#4C78A8",ax=ax)
  ax.set(title=f"{sex.title()} – Hạng vé {pc}",xlabel="Cảng",ylabel="Tỷ lệ sống (%)" if j==0 else ""); ax.set_ylim(0,100)
fig.suptitle("Tỷ lệ sống sót theo cảng, hạng vé, giới tính"); plt.tight_layout()
print(f"Biểu đồ phân tầng | {STUDENT_INFO}"); plt.show()
print("Nhận xét: nữ và hành khách hạng vé 1 nhìn chung có tỷ lệ sống cao hơn; cần so sánh cảng trong từng nhóm giới tính/hạng vé.")


## Câu 5. Người đi cùng và sống sót


In [ ]:
q=df.copy(); q["family_size"]=q.sibsp+q.parch+1
q["family_group"]=pd.cut(q.family_size,[0,1,4,np.inf],labels=["Một mình","Gia đình nhỏ (2–4)","Gia đình lớn (5+)"])
cnt=q.family_size.value_counts().sort_index().rename_axis("Số người nhóm").to_frame("Số hành khách")
rate=q.groupby("family_group",observed=False).survived.mean().mul(100).round(1).rename("Tỷ lệ sống sót (%)").to_frame()
print(f"Số hành khách theo số người đi cùng (gồm bản thân) | {STUDENT_INFO}"); display(cnt)
print(f"Tỷ lệ sống theo quy mô gia đình | {STUDENT_INFO}"); display(rate)
fig,ax=plt.subplots(figsize=(9,5)); sns.barplot(data=q,x="family_group",y="survived",order=["Một mình","Gia đình nhỏ (2–4)","Gia đình lớn (5+)"],color="#59A14F",ax=ax)
ax.set(title="Quy mô nhóm và tỷ lệ sống sót",xlabel="Quy mô nhóm",ylabel="Tỷ lệ sống"); ax.set_ylim(0,1); plt.xticks(rotation=10); plt.tight_layout()
print(f"Biểu đồ quy mô gia đình | {STUDENT_INFO}"); plt.show()
print("Nhận xét: tỷ lệ sống khác nhau theo quy mô nhóm; đây là liên hệ mô tả, không khẳng định nhân quả.")


## Câu 6. Xử lý thiếu và bỏ thuộc tính

Điền tuổi/giá vé bằng trung vị, cảng bằng mode. Bỏ cabin, boat, body, home.dest, ticket (nhiều thiếu/không cần ở các bước sau; boat/body phản ánh thông tin sau sự kiện). Giữ name đến câu 7.


In [ ]:
clean_df=df.copy()
clean_df["age"]=clean_df.age.fillna(clean_df.age.median()); clean_df["fare"]=clean_df.fare.fillna(clean_df.fare.median())
clean_df["embarked"]=clean_df.embarked.fillna(clean_df.embarked.mode().iloc[0])
clean_df=clean_df.drop(columns=["cabin","boat","body","home.dest","ticket"],errors="ignore")
print(f"Kích thước sau xử lý: {clean_df.shape} | {STUDENT_INFO}")
print(f"Giá trị rỗng còn lại | {STUDENT_INFO}"); display(clean_df.isna().sum().to_frame("Số giá trị rỗng"))


## Câu 7. Biến đổi thuộc tính


In [ ]:
clean_df["title"]=clean_df.name.str.extract(r",\s*([^.]*)\.",expand=False).str.strip(); clean_df=clean_df.drop(columns=["name"])
fare_for_bins=clean_df["fare"].copy()
clean_df["fare"]=clean_df.fare.round().astype("int64")
clean_df["sex"]=clean_df.sex.map({"male":0,"female":1}).astype(int); clean_df["embarked"]=clean_df.embarked.map({"C":0,"Q":1,"S":2}).astype(int)
clean_df["age_group"]=pd.cut(clean_df.age,[-np.inf,11,18,22,27,33,40,66,np.inf],labels=range(1,9),right=True,include_lowest=True).astype(int)
clean_df["fare_group"]=pd.cut(fare_for_bins,[-np.inf,7.91,14.454,31,99,250,np.inf],labels=range(1,7),right=True,include_lowest=True).astype(int)
groups=pd.concat([clean_df.age_group.value_counts().sort_index().rename("Mẫu nhóm tuổi"),clean_df.fare_group.value_counts().sort_index().rename("Mẫu nhóm giá vé")],axis=1).fillna(0).astype(int)
print(f"Số mẫu mỗi nhóm tuổi/giá vé (theo các ngưỡng đề bài) | {STUDENT_INFO}"); display(groups)
print(f"Dữ liệu sau biến đổi – 5 dòng đầu | {STUDENT_INFO}"); display(clean_df.head())


## Câu 8. Tạo thuộc tính mới


In [ ]:
clean_df["age_class"]=clean_df.age*clean_df.pclass
clean_df["fare_per_person"]=clean_df.fare/(clean_df.sibsp+clean_df.parch+1)
print(f"Dữ liệu có thuộc tính mới | {STUDENT_INFO}"); display(clean_df[["age","pclass","age_class","fare","sibsp","parch","fare_per_person"]].head())
print(f"Thống kê thuộc tính mới | {STUDENT_INFO}"); display(clean_df[["age_class","fare_per_person"]].describe().T)
